### Restricted-data analysis not included

This block used participant-level clinical and baseline data. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


### Restricted-data analysis not included

This block used participant-level clinical and baseline data. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


### Restricted-data analysis not included

This block used participant-level baseline clinical data. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


### Restricted-data analysis not included

This block used participant-level clinical data. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


### Restricted-data analysis not included

This block used participant-level clinical subgroup data. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


### Restricted-data analysis not included

This block used participant-level clinical data. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


### Restricted-data analysis not included

This block used participant-level clinical data. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


### Restricted-data analysis not included

This block used participant-level clinical modeling and baseline cohort data. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


### Restricted-data analysis not included

This block used participant-level clinical modeling and baseline cohort data. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


In [ ]:
from pathlib import Path
import pandas as pd

CHD_ISTROKE = Path("/home/qinbw/CIPHER/fig5_final/outputs_final_valid_select_more_models_ASCVD_toprisk/ALL_model_results_long.csv")
ASCVD = Path("/home/qinbw/CIPHER/fig5_final/outputs_final_valid_select_more_models_ASCVD_toprisk_ASCVD/ALL_model_results_long.csv")

OUT = Path("/home/qinbw/CIPHER/fig5_final/outputs_final_valid_select_more_models_ASCVD_toprisk/ALL_model_results_long_merged_CHD_istroke_ASCVD.csv")

df1 = pd.read_csv(CHD_ISTROKE)
df2 = pd.read_csv(ASCVD)

df1["source_dir"] = "CHD_istroke"
df2["source_dir"] = "ASCVD"

if "endpoint" in df2.columns:
    df2["endpoint"] = df2["endpoint"].replace({
        "ICVD": "ASCVD",
        "icvd": "ASCVD",
        "ascvd": "ASCVD",
    })

df = pd.concat([df1, df2], ignore_index=True, sort=False)

key_cols = [c for c in ["split", "endpoint", "bootstrap", "strategy", "model", "target"] if c in df.columns]
if key_cols:
    df = df.drop_duplicates(subset=key_cols, keep="last")

df.to_csv(OUT, index=False)

print("Saved:", OUT)
print("Shape:", df.shape)
print(df["endpoint"].value_counts(dropna=False))

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from __future__ import annotations

import os
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

# ======================================================
# Global settings
# ======================================================
warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

sns.set_style("white")


# ======================================================
# Config
# ======================================================
ROOT = Path("/home/qinbw/CIPHER/fig5_final")
IN_DIR = ROOT / "outputs_final_valid_select_more_models_ASCVD_toprisk"
OUT_DIR = IN_DIR / "boxplot_redraw"
OUT_DIR.mkdir(exist_ok=True, parents=True)

# 优先读总结果；如果没有就自己改成 ASCVD_model_results_long.csv 等
RESULT_CSV = IN_DIR / "ALL_model_results_long_merged_CHD_istroke_ASCVD.csv"

# endpoint 顺序
ENDPOINT_ORDER = ["CHD", "istroke", "ASCVD"]

# ------------------------------------------------------
# 颜色：你给的5个 + 1个莫兰迪色
# ------------------------------------------------------
MODEL_COLORS = {
    "Clinical → normalized risk": "#4C72B0",
    "Clinical → risk → Cox": "#55A868",
    "Clinical → top-risk label": "#8172B2",
    "Clinical → time-to-event": "#DD8452",
    "Clinical → 5y event": "#C44E52",
    "Clinical Cox": "#9C8C98",   # 莫兰迪配色
}
MODEL_COLORS = {
    # brighter red/orange Morandi
    "Clinical → normalized risk": "#D27A73",   # brighter muted rose red
    "Clinical → risk → Cox": "#C98A5A",        # brighter muted orange
    "Clinical → top-risk label": "#B9786A",    # brighter muted terracotta

    # gray Morandi
    "Clinical → time-to-event": "#70777F",
    "Clinical → 5y event": "#85808A",
    "Clinical Cox": "#969089",
}

# ------------------------------------------------------
# 左右顺序：risk 在左，direct 在右
# ------------------------------------------------------
RISK_METHODS = [
    "Clinical → normalized risk",
    "Clinical → risk → Cox",
    "Clinical → top-risk label",
]

DIRECT_METHODS = [
    "Clinical → time-to-event",
    "Clinical → 5y event",
    "Clinical Cox",
]

BOX_ORDER = RISK_METHODS + DIRECT_METHODS

# ------------------------------------------------------
# legend 顺序：direct 在上，risk 在下
# ------------------------------------------------------
LEGEND_ORDER = DIRECT_METHODS + RISK_METHODS

def add_group_mean_annotations(
    ax,
    d: pd.DataFrame,
    metric: str,
    methods_present: list[str],
    box_width: float = 0.75,
    line_color: str = "#C44E52",
    line_lw: float = 1.2,
    line_alpha: float = 0.95,
):
    """
    For each endpoint tick:
    - left risk-based boxes: dashed red mean line
    - right direct-based boxes: dashed red mean line
    - text above all boxes: risk mean - direct mean
    """

    d = d.copy()
    d = d[np.isfinite(d[metric])].copy()

    if d.empty:
        return

    # 当前图中实际出现的方法
    methods_present = [m for m in methods_present if m in d["method_pretty"].dropna().unique()]
    n_hue = len(methods_present)

    if n_hue == 0:
        return

    # seaborn dodge box 的近似位置
    step = box_width / n_hue
    offsets = {
        m: -box_width / 2 + step / 2 + i * step
        for i, m in enumerate(methods_present)
    }

    endpoints = [e for e in ENDPOINT_ORDER if e in d["endpoint"].astype(str).unique()]

    y_min = np.nanmin(d[metric].values)
    y_max = np.nanmax(d[metric].values)
    y_range = max(y_max - y_min, 0.02)

    # 给上方文字留空间
    ax.set_ylim(y_min - 0.03 * y_range, y_max + 0.18 * y_range)

    for x_idx, ep in enumerate(endpoints):
        dd = d[d["endpoint"].astype(str) == ep].copy()
        if dd.empty:
            continue

        risk_dd = dd[dd["method_pretty"].isin(RISK_METHODS)]
        direct_dd = dd[dd["method_pretty"].isin(DIRECT_METHODS)]

        if risk_dd.empty or direct_dd.empty:
            continue

        risk_mean = risk_dd[metric].mean()
        direct_mean = direct_dd[metric].mean()

        delta = risk_mean - direct_mean
        rel = delta / direct_mean * 100 if np.isfinite(direct_mean) and direct_mean != 0 else np.nan

        # 计算 risk/direct 三个 box 的 x 范围
        risk_methods_here = [m for m in RISK_METHODS if m in methods_present]
        direct_methods_here = [m for m in DIRECT_METHODS if m in methods_present]

        if len(risk_methods_here) > 0:
            x_risk_min = x_idx + min(offsets[m] for m in risk_methods_here) - step * 0.35
            x_risk_max = x_idx + max(offsets[m] for m in risk_methods_here) + step * 0.35

            ax.hlines(
                y=risk_mean,
                xmin=x_risk_min,
                xmax=x_risk_max,
                colors=line_color,
                linestyles="--",
                linewidth=line_lw,
                alpha=line_alpha,
                zorder=5,
            )

        if len(direct_methods_here) > 0:
            x_direct_min = x_idx + min(offsets[m] for m in direct_methods_here) - step * 0.35
            x_direct_max = x_idx + max(offsets[m] for m in direct_methods_here) + step * 0.35

            ax.hlines(
                y=direct_mean,
                xmin=x_direct_min,
                xmax=x_direct_max,
                colors=line_color,
                linestyles="--",
                linewidth=line_lw,
                alpha=line_alpha,
                zorder=5,
            )

        # 上方标注
        y_text = max(dd[metric].max(), risk_mean, direct_mean) + 0.07 * y_range

        if np.isfinite(rel):
            text = f"Δ={delta:+.3f}\n({rel:+.1f}%)"
        else:
            text = f"Δ={delta:+.3f}"

        ax.text(
            x_idx,
            y_text,
            text,
            ha="center",
            va="bottom",
            fontsize=9,
            color=line_color,
        )

# ======================================================
# Helpers
# ======================================================
def prettify_method(strategy: str) -> str:
    """
    把原始 strategy 映射成图上的方法名称
    """
    mapping = {
        "risk_regression": "Clinical → normalized risk",
        "risk_cox": "Clinical → risk → Cox",
        "top_risk_label_classification": "Clinical → top-risk label",
        "direct_survival_regression": "Clinical → time-to-event",
        "direct_event_classification": "Clinical → 5y event",
        "clinical_cox": "Clinical Cox",
    }
    return mapping.get(strategy, strategy)


def load_results(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # 方法名
    df["method_pretty"] = df["strategy"].map(prettify_method)

    # endpoint 顺序
    df["endpoint"] = pd.Categorical(
        df["endpoint"],
        categories=[x for x in ENDPOINT_ORDER if x in df["endpoint"].unique()],
        ordered=True
    )

    # 方法顺序
    existing_methods = [m for m in BOX_ORDER if m in df["method_pretty"].unique()]
    df["method_pretty"] = pd.Categorical(
        df["method_pretty"],
        categories=existing_methods,
        ordered=True
    )

    return df


def make_custom_legend(ax, methods_present):
    """
    legend中 direct 在上，risk 在下
    """
    legend_methods = [m for m in LEGEND_ORDER if m in methods_present]

    handles = [
        Patch(facecolor=MODEL_COLORS[m], edgecolor="black", label=m)
        for m in legend_methods
    ]

    ax.legend(
        handles=handles,
        labels=legend_methods,
        title="",
        frameon=False,
        bbox_to_anchor=(1.02, 1.0),
        loc="upper left",
        borderaxespad=0.0,
    )


def plot_boxplot(
    df: pd.DataFrame,
    split: str,
    metric: str,
    out_file: Path,
    y_label: str | None = None,
    ylim: tuple[float, float] | None = None,
):
    d = df[df["split"] == split].copy()
    d = d[np.isfinite(d[metric])].copy()

    if d.empty:
        print(f"[WARN] No data for split={split}, metric={metric}")
        return

    methods_present = [m for m in BOX_ORDER if m in d["method_pretty"].unique()]

    plt.figure(figsize=(12, 6.5))

    ax = sns.boxplot(
        data=d,
        x="endpoint",
        y=metric,
        hue="method_pretty",
        hue_order=methods_present,   # 左右顺序：risk 左，direct 右
        palette={m: MODEL_COLORS[m] for m in methods_present},
        showfliers=False,
        width=0.75,
        linewidth=1.0,
    )
    
    # stripplot 会重复生成 legend，这里先删掉默认 legend
    if ax.legend_ is not None:
        ax.legend_.remove()
    add_group_mean_annotations(
        ax=ax,
        d=d,
        metric=metric,
        methods_present=methods_present,
        box_width=0.75,
    )
    '''
    # 自定义 legend：direct 在上，risk 在下
    make_custom_legend(ax, methods_present)
    '''
    ax.set_xlabel("")
    ax.set_ylabel(metric if y_label is None else y_label, fontsize=12)

    if ylim is not None:
        ax.set_ylim(*ylim)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    '''
    # 在 x 轴下面加一行说明
    ax.text(
        0.5, -0.18,
        "Within each endpoint: risk-based methods on the left, direct-outcome methods on the right",
        transform=ax.transAxes,
        ha="center", va="top", fontsize=10
    )
    '''
    plt.tight_layout()
    plt.savefig(out_file, dpi=300, transparent=True, bbox_inches="tight")
    plt.close()

    print(f"[Saved] {out_file}")


def plot_four_panel(df: pd.DataFrame, out_file: Path):
    """
    可选：做一个 2x2 总图
    左上：valid cindex
    右上：test cindex
    左下：valid auc_5y
    右下：test auc_5y
    """
    fig, axes = plt.subplots(2, 2, figsize=(18, 11))
    axes = axes.flatten()

    panel_cfg = [
        ("valid", "cindex", "Validation C-index"),
        ("test", "cindex", "External test C-index"),
        ("valid", "auc_5y", "Validation AUC at 5 years"),
        ("test", "auc_5y", "External test AUC at 5 years"),
    ]

    for ax, (split, metric, title) in zip(axes, panel_cfg):
        d = df[df["split"] == split].copy()
        d = d[np.isfinite(d[metric])].copy()

        if d.empty:
            ax.set_axis_off()
            continue

        methods_present = [m for m in BOX_ORDER if m in d["method_pretty"].unique()]

        sns.boxplot(
            data=d,
            x="endpoint",
            y=metric,
            hue="method_pretty",
            hue_order=methods_present,
            palette={m: MODEL_COLORS[m] for m in methods_present},
            showfliers=False,
            width=0.75,
            linewidth=1.0,
            ax=ax,
        )

        sns.stripplot(
            data=d,
            x="endpoint",
            y=metric,
            hue="method_pretty",
            hue_order=methods_present,
            dodge=True,
            color="black",
            size=1.8,
            alpha=0.18,
            ax=ax,
        )

        if ax.legend_ is not None:
            ax.legend_.remove()
        add_group_mean_annotations(
            ax=ax,
            d=d,
            metric=metric,
            methods_present=methods_present,
            box_width=0.75,
        )
        ax.set_title(title, fontsize=13)
        ax.set_xlabel("")
        ax.set_ylabel(metric, fontsize=11)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # 只在整张图右边放一个总 legend
    methods_present_all = [m for m in LEGEND_ORDER if m in df["method_pretty"].unique()]
    handles = [
        Patch(facecolor=MODEL_COLORS[m], edgecolor="black", label=m)
        for m in methods_present_all
    ]

    fig.legend(
        handles=handles,
        labels=methods_present_all,
        frameon=False,
        bbox_to_anchor=(1.02, 0.98),
        loc="upper left",
        title="",
    )

    plt.tight_layout()
    plt.savefig(out_file, dpi=300, transparent=True, bbox_inches="tight")
    plt.close()

    print(f"[Saved] {out_file}")


# ======================================================
# Main
# ======================================================
if __name__ == "__main__":

    if not RESULT_CSV.exists():
        raise FileNotFoundError(f"Result file not found: {RESULT_CSV}")

    df = load_results(RESULT_CSV)

    print("Loaded:", RESULT_CSV)
    print("Shape:", df.shape)
    print("Endpoints:", df["endpoint"].dropna().unique())
    print("Methods:", df["method_pretty"].dropna().unique())

    # 单独四张图
    plot_boxplot(
        df=df,
        split="valid",
        metric="cindex",
        out_file=OUT_DIR / "valid_cindex_boxplot_reordered.pdf",
        y_label="C-index",
    )

    plot_boxplot(
        df=df,
        split="test",
        metric="cindex",
        out_file=OUT_DIR / "test_cindex_boxplot_reordered.pdf",
        y_label="C-index",
    )

    plot_boxplot(
        df=df,
        split="valid",
        metric="auc_5y",
        out_file=OUT_DIR / "valid_auc5y_boxplot_reordered.pdf",
        y_label="AUC at 5 years",
    )

    plot_boxplot(
        df=df,
        split="test",
        metric="auc_5y",
        out_file=OUT_DIR / "test_auc5y_boxplot_reordered.pdf",
        y_label="AUC at 5 years",
    )

    # 再给一张 2x2 总图
    plot_four_panel(
        df=df,
        out_file=OUT_DIR / "boxplot_summary_2x2.pdf",
    )

    print("\nDONE")
    print("Output dir:", OUT_DIR)

### Restricted-data analysis not included

This block used local notebook shell search. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.


### Restricted-data analysis not included

This block used participant-level baseline and modality-membership data. The public workflow starts from its de-identified derived output; no raw-data access or processing code is included in this release.
